# TartanAir 640 — kiểm chứng ba dataset shard (đếm file thật, so MD5 với nguồn)

Notebook này **không** tin vào danh sách hiển thị trên giao diện. Nó gắn ba dataset rồi:
1. **đếm** mọi file và byte có thật trên đĩa của từng dataset;
2. so **từng quỹ đạo** với báo cáo lúc dựng (`shard<N>.json`) và kế hoạch (`plan.json`): số khung, byte ảnh,
   `frames.npy`, IMU, `build_done.json`;
3. gộp ba shard như notebook train (`find_shard_roots` + `link_trajectories`), dựng manifest với
   `split_rule: per_environment`, kiểm **mọi** cặp ảnh–IMU: ảnh thứ i phải là khung `frames.npy[i]`, có
   cửa sổ IMU 128 mẫu đặt giữa nó, không khung nào bị bỏ;
4. so **MD5 của mọi file** với nguồn: shard 1, 2 với `inventory_shard<N>.tsv` (MD5 tính từ Output gốc lúc đóng
   gói); shard 0 với 24 file tar trong Output của `qwt-jepa-ver11` (nguồn của `tartanairshard0`).

Kết quả: `/kaggle/working/verify_report.json` và dòng cuối **KHỚP** / **KHÔNG KHỚP**.

## Chạy
*New Notebook* → *Import Notebook* → file này. *Settings*: **Accelerator: None**, **Internet: On**. Sửa
`DATASETS` nếu bạn đặt tên dataset khác. Chạy trong **phiên mới** (Cell 3 phải thấy đúng ba dataset dưới
`/kaggle/input`; Cell 4 mới gắn thêm Output nguồn của shard 0). `CONTENT = True` đọc ~110 GB: ~30–60 phút CPU.

## Cell 1 — Source, gắn ba dataset

In [ ]:
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import json
import subprocess
import sys
import time

DATASETS = {0: 'buidinhkhoi/tartanairshard0', 1: 'buidinhkhoi/tartanair640-s1', 2: 'buidinhkhoi/tartanair640-s2'}
SHARD0_SOURCE = ('buidinhkhoi/qwt-jepa-ver11', 7)  # Output có 24 tar mà tartanairshard0 được tạo từ đó
CONTENT = True   # MD5 mọi file của dataset và nguồn
REPO_URL = 'https://github.com/leesanghyoek/qwt_jepa_ver5.git'
SOURCE_REF = '8bbd447'
PROJECT = Path('/tmp/qwt_jepa_version3')
REPORT = Path('/kaggle/working/verify_report.json')

if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, str(PROJECT)], check=True)
subprocess.run(['git', '-C', str(PROJECT), 'fetch', '--quiet', 'origin'], check=True)
subprocess.run(['git', '-C', str(PROJECT), 'checkout', '--quiet', '--detach', SOURCE_REF], check=True)
for path in (str(PROJECT / 'tools'), str(PROJECT)):
    if path not in sys.path:
        sys.path.insert(0, path)
for name in [m for m in sys.modules if m == 'build_tartanair640' or m == 'qjepa' or m.startswith('qjepa.')]:
    del sys.modules[name]
import build_tartanair640 as builder
import kagglehub

roots = {shard: Path(kagglehub.dataset_download(handle)) for shard, handle in DATASETS.items()}
for shard, root in roots.items():
    print('shard', shard, DATASETS[shard], '→', root)
results = {shard: {'dataset': DATASETS[shard], 'root': str(root)} for shard, root in roots.items()}

## Cell 2 — Đếm file thật và so từng quỹ đạo với báo cáo lúc dựng

In [ ]:
plans = {}
for shard, root in roots.items():
    started = time.perf_counter()
    stats = builder.tree_stats(root)
    report = json.loads((root / 'build_meta' / f'shard{shard}.json').read_text())
    plans[shard] = plan = json.loads((root / 'build_meta' / 'plan.json').read_text())
    planned = {name: row for name, row in plan['environments'].items() if row['shard'] == shard}
    check = builder.check_shard(root, report)
    expected = (len(planned), sum(row['trajectories'] for row in planned.values()),
                sum(row['frames_kept'] for row in planned.values()))
    results[shard].update(
        files=stats['files'], bytes=stats['bytes'], top=stats['top'], fingerprint=report['fingerprint'],
        environments=check['environments'], trajectories=check['trajectories'], frames=check['frames'],
        image_bytes=check['image_bytes'], planned=dict(zip(('environments', 'trajectories', 'frames'), expected)),
        problems=len(check['problems']), problem_examples=check['problems'][:10],
        structure_ok=not check['problems'] and (check['environments'], check['trajectories'], check['frames']) == expected
        and check['image_bytes'] == report['stored_bytes'])
    print(f"shard {shard}: {stats['files']:,} file, {stats['bytes'] / 1e9:.3f} GB thật trên đĩa "
          f"| {check['environments']}/{expected[0]} môi trường, {check['trajectories']}/{expected[1]} quỹ đạo, "
          f"{check['frames']:,}/{expected[2]:,} khung, {check['image_bytes'] / 1e9:.3f} GB ảnh "
          f"| lỗi {len(check['problems'])} | {time.perf_counter() - started:.0f} s")
    for name, row in stats['top'].items():
        print(f"      {name:24s} {row['files']:7,} file {row['bytes'] / 1e9:8.3f} GB")
    for problem in check['problems'][:10]:
        print('   LỖI', problem)
assert len({plan['fingerprint'] for plan in plans.values()}) == 1, 'Ba shard không cùng một kế hoạch.'

## Cell 3 — Gộp ba shard như notebook train, dựng manifest, kiểm mọi cặp ảnh–IMU

Đây đúng là đường notebook train đi khi `TARTANAIR640 = True`: `find_shard_roots('/kaggle/input')` →
`link_trajectories` → manifest với `per_environment` và seed/cửa sổ của `configs/kaggle_infomax.yaml`.

In [ ]:
import numpy as np
from PIL import Image
from qjepa.config import load_config, serializable_config
from qjepa.data.manifest import build_manifest
from qjepa.data.tartanair import find_shard_roots, link_trajectories

started = time.perf_counter()
results['merged'] = merged_result = {'ok': False}
try:
    found = find_shard_roots('/kaggle/input')
    assert {s: p.resolve() for s, p in found.items()} == {s: p.resolve() for s, p in roots.items()}, found
    merged = link_trajectories(list(found.values()), Path('/tmp/tartanair640_all'))
    data = serializable_config(load_config(PROJECT / 'configs/kaggle_infomax.yaml'))['data']
    manifest = build_manifest(merged, window=data['imu_window'], seed=data['corruption_seed'], split_rule='per_environment')
    meta = manifest['meta']
    rejected = {key: sum(item[key] for item in meta['pairing_audit'])
                for key in ('rejected_outside', 'rejected_centre', 'rejected_nonuniform')}
    plan = next(iter(plans.values()))
    planned_frames = sum(row['frames_kept'] for row in plan['environments'].values())
    planned_trajectories = sum(row['trajectories'] for row in plan['environments'].values())
    samples = [sample for split in manifest['samples'].values() for sample in split]

    # Ảnh thứ i của quỹ đạo phải là khung frames.npy[i]: cam_time hàng i là thời điểm của đúng khung đó.
    kept = {}
    mispaired = 0
    for sample in samples:
        if sample.trajectory_path not in kept:
            kept[sample.trajectory_path] = np.load(Path(sample.trajectory_path) / 'frames.npy')
        mispaired += int(Path(sample.image_path).name.split('_', 1)[0]) != int(kept[sample.trajectory_path][sample.image_index])
    mounts = tuple(str(root.resolve()) + '/' for root in roots.values())
    outside = sum(not sample.image_path.startswith(mounts) for sample in samples)  # symlinks resolved to the datasets
    rng = np.random.default_rng(0)
    sizes = Counter()
    for index in rng.choice(len(samples), min(96, len(samples)), replace=False):
        with Image.open(samples[index].image_path) as image:
            sizes[image.convert('RGB').size] += 1

    groups = {split: len({(s.environment, s.trajectory_id) for s in rows}) for split, rows in manifest['samples'].items()}
    environments = {split: len({s.environment for s in rows}) for split, rows in manifest['samples'].items()}
    trajectories = sum(len(keys) for keys in meta['trajectories_per_split'].values())
    merged_result.update({
        'trajectories': trajectories, 'planned_trajectories': planned_trajectories,
        'samples': len(samples), 'planned_frames': planned_frames, 'rejected': rejected,
        'samples_per_split': meta['samples_per_split'], 'groups_per_split': groups,
        'environments_per_split': environments, 'mispaired': mispaired, 'outside_input': outside,
        'decoded': {str(key): count for key, count in sizes.items()}, 'manifest_hash': meta['manifest_hash'],
    })
    merged_result['ok'] = (trajectories == planned_trajectories and len(samples) == planned_frames
                           and not any(rejected.values()) and mispaired == 0 and outside == 0
                           and set(sizes) == {(640, 640)})
    print(json.dumps(merged_result, indent=1))
except (ValueError, FileNotFoundError) as error:  # một shard thiếu hay hỏng: ghi lại, vẫn chạy tiếp
    merged_result['error'] = f'{type(error).__name__}: {error}'
    print('LỖI', merged_result['error'])
print(f'{time.perf_counter() - started:.0f} s')

## Cell 4 — MD5 của mọi file so với nguồn

Shard 1, 2: so với `inventory_shard<N>.tsv`, MD5 tính từ chính Output gốc khi đóng gói. Shard 0: gắn Output
`qwt-jepa-ver11` (24 tar), đọc MD5 từng file trong tar và so với file đã giải nén trong `tartanairshard0`.

In [ ]:
if CONTENT:
    for shard, root in roots.items():
        started = time.perf_counter()
        inventory = root / 'build_meta' / f'inventory_shard{shard}.tsv'
        if inventory.is_file():
            expected, source = builder.read_inventory(inventory), inventory.name
        else:
            notebook, version = SHARD0_SOURCE
            source_root = Path(kagglehub.notebook_output_download(f'{notebook}/versions/{version}'))
            tars = sorted(source_root.rglob('*.tar'))
            assert tars, f'{notebook} v{version}: không có file tar'
            expected = {}
            with ThreadPoolExecutor(4) as executor:
                for entries in executor.map(builder.archive_inventory, tars):
                    expected.update(entries)
            source = f'{notebook} v{version}: {len(tars)} tar'
        comparison = builder.compare_inventories(expected, builder.file_inventory(root))
        comparison['source'] = source
        comparison['source_bytes'] = sum(size for size, _ in expected.values())
        results[shard]['content'] = comparison
        print(f"shard {shard}: nguồn {source} {comparison['expected']:,} file | dataset {comparison['actual']:,} | "
              f"thiếu {comparison['missing']}, thừa {comparison['extra']}, khác MD5 {comparison['different']} "
              f"| {time.perf_counter() - started:.0f} s")
        for example in comparison['examples']:
            print('   ', example)

## Cell 5 — Kết luận

In [ ]:
ok = results['merged']['ok'] and all(results[shard]['structure_ok'] for shard in roots)
if CONTENT:
    ok = ok and all(results[shard]['content']['match'] for shard in roots)
results['ok'] = ok
REPORT.write_text(json.dumps(results, indent=2, ensure_ascii=False))
print(f"{'shard':6s} {'file thật':>10s} {'GB':>8s} {'quỹ đạo':>8s} {'khung':>8s} {'cấu trúc':>9s} {'MD5':>6s}")
for shard in roots:
    row = results[shard]
    content = row.get('content', {}).get('match', 'bỏ qua')
    print(f"{shard:<6d} {row['files']:10,} {row['bytes'] / 1e9:8.3f} {row['trajectories']:8d} {row['frames']:8,} "
          f"{str(row['structure_ok']):>9s} {str(content):>6s}")
merged_result = results['merged']
print('gộp:', merged_result.get('error') or f"{merged_result['trajectories']} quỹ đạo, {merged_result['samples']:,} "
      f"cặp ảnh–IMU, nhóm Pxxx theo split {merged_result['groups_per_split']}")
print('\nKHỚP — dùng được để train.' if ok else '\nKHÔNG KHỚP — xem các dòng LỖI ở trên; đừng train trên bộ này.')